In [4]:
import os
import pandas as pd
import numpy as np
from sklearn.feature_selection import (
    SelectKBest, chi2, f_classif, mutual_info_classif, VarianceThreshold
)
from sklearn.preprocessing import MinMaxScaler
import warnings

warnings.filterwarnings("ignore")

INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\featuring"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Función para aplicar selección de variables
def seleccionar_variables(X_train, y_train, metodo, k=100):
    if metodo == "CHI2":
        X_train_nonneg = MinMaxScaler().fit_transform(X_train)
        scores = chi2(X_train_nonneg, y_train)[0]
    elif metodo == "ANOVA":
        scores = f_classif(X_train, y_train)[0]
    elif metodo == "MI":
        scores = mutual_info_classif(X_train, y_train)
    elif metodo == "VAR":
        selector = VarianceThreshold()
        selector.fit(X_train)
        mask = selector.get_support()
        cols = X_train.columns[mask]
        return cols, pd.Series(selector.variances_, index=X_train.columns)
    else:
        raise ValueError("Método de selección no reconocido")

    # Selección dinámica por umbral (evita K fijo)
    threshold = 0.05 * np.nanmax(scores)
    selected_mask = scores >= threshold
    selected_features = X_train.columns[selected_mask]
    scores_series = pd.Series(scores, index=X_train.columns)

    return selected_features, scores_series

# Procesar todas las combinaciones
metodos = ["CHI2", "ANOVA", "MI", "VAR", "ALL"]
resumen_comparativo = []

for carpeta in os.listdir(INPUT_PATH):
    carpeta_path = os.path.join(INPUT_PATH, carpeta)
    if not os.path.isdir(carpeta_path):
        continue

    for tipo in ["ORIGINAL", "FE"]:
        nombre_dataset = f"{carpeta}_{tipo}"
        try:
            X_train = pd.read_parquet(f"{carpeta_path}/X_train_{carpeta}_{tipo}.parquet")
            X_test = pd.read_parquet(f"{carpeta_path}/X_test_{carpeta}_{tipo}.parquet")
            y_train = pd.read_parquet(f"{carpeta_path}/y_train_{carpeta}_{tipo}.parquet")
            y_test = pd.read_parquet(f"{carpeta_path}/y_test_{carpeta}_{tipo}.parquet")
        except Exception as e:
            print(f"❌ Error cargando {nombre_dataset}: {e}")
            continue

        print(f"\n🔍 Procesando {nombre_dataset}...")

        for metodo in metodos:
            if metodo == "CHI2" and (X_train < 0).any().any():
                print(f"⚠️ Saltando CHI2 por contener negativos en {nombre_dataset}")
                continue

            if metodo == "ALL":
                selected_columns = X_train.columns
                scores = pd.Series([1.0] * len(selected_columns), index=selected_columns)
            else:
                try:
                    selected_columns, scores = seleccionar_variables(X_train, y_train.values.ravel(), metodo)
                except Exception as e:
                    print(f"❌ Error aplicando {metodo} en {nombre_dataset}: {e}")
                    continue

            # Guardar métricas
            metricas_path = os.path.join(OUTPUT_PATH, f"metricas_{nombre_dataset}_{metodo}.csv")
            try:
                scores.to_csv(metricas_path, header=["score"])
                print(f"📅 Métricas guardadas: {metricas_path}")
            except Exception as e:
                print(f"❌ Error al guardar métricas de {metodo} en {nombre_dataset}: {e}")

            # Guardar datasets filtrados
            X_train_sel = X_train[selected_columns]
            X_test_sel = X_test[selected_columns.intersection(X_test.columns)]

            X_train_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_train_{nombre_dataset}_{metodo}.parquet"))
            X_test_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_test_{nombre_dataset}_{metodo}.parquet"))
            y_train.to_parquet(os.path.join(OUTPUT_PATH, f"y_train_{nombre_dataset}_{metodo}.parquet"))
            y_test.to_parquet(os.path.join(OUTPUT_PATH, f"y_test_{nombre_dataset}_{metodo}.parquet"))

            print(f"✔ {nombre_dataset} - {metodo}: {len(selected_columns)} variables seleccionadas")

            resumen_comparativo.append({
                "dataset": nombre_dataset,
                "metodo": metodo,
                "columnas_seleccionadas": len(selected_columns)
            })

# Guardar resumen comparativo
df_resumen = pd.DataFrame(resumen_comparativo)
df_resumen.to_csv(os.path.join(OUTPUT_PATH, "resumen_cantidad_variables.csv"), index=False)
print("\n📊 Comparativo guardado en resumen_cantidad_variables.csv")



🔍 Procesando MaxAbs_ORIGINAL...
⚠️ Saltando CHI2 por contener negativos en MaxAbs_ORIGINAL
📅 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_MaxAbs_ORIGINAL_ANOVA.csv
✔ MaxAbs_ORIGINAL - ANOVA: 118 variables seleccionadas
📅 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_MaxAbs_ORIGINAL_MI.csv
✔ MaxAbs_ORIGINAL - MI: 128 variables seleccionadas
📅 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_MaxAbs_ORIGINAL_VAR.csv
✔ MaxAbs_ORIGINAL - VAR: 539 variables seleccionadas
📅 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\selected\metricas_MaxAbs_ORIGINAL_ALL.csv
✔ MaxAbs_ORIGINAL - ALL: 539 variables seleccionadas

🔍 Procesando MaxAbs_FE...
⚠️ Saltando CHI2 por contener negativos en MaxAbs_FE
📅 Métricas guardadas: C:\Users\Gonzalo\Documents\GitHub\Te